## データ前処理
- bronze
- silver

### ブロンズ

In [0]:
CATALOG = "workspace"
SCHEMA = "bank"

CUSTOMER_PATH = "/Volumes/workspace/bank/vol/rdb/customer.csv"
TRANSACTION_PATH = "/Volumes/workspace/bank/vol/rdb/transaction_summary.csv"
CRM_PATH = "/Volumes/workspace/bank/vol/crm/crm_activity.csv"

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bank/vol/rdb"))
display(dbutils.fs.ls("/Volumes/workspace/bank/vol/crm"))

In [0]:
bronze_customer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMER_PATH)
)

display(bronze_customer_df)

In [0]:
(
    bronze_customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.bronze_customer")
)

In [0]:
bronze_transaction_summary_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(TRANSACTION_PATH)
)

display(bronze_transaction_summary_df)

In [0]:
(
    bronze_transaction_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.bronze_transaction_summary")
)

In [0]:
bronze_crm_activity_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CRM_PATH)
)

display(bronze_crm_activity_df)

In [0]:
(
    bronze_crm_activity_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.bronze_crm_activity")
)

In [0]:
%sql
SHOW TABLES IN workspace.bank;

In [0]:
%sql
SELECT
    'bronze_customer' AS table_name,
    COUNT(*) AS row_count
FROM workspace.bank.bronze_customer

UNION ALL

SELECT
    'bronze_transaction_summary',
    COUNT(*)
FROM workspace.bank.bronze_transaction_summary

UNION ALL

SELECT
    'bronze_crm_activity',
    COUNT(*)
FROM workspace.bank.bronze_crm_activity;

### シルバー

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, IntegerType

bronze_customer_df = spark.table(
    "workspace.bank.bronze_customer"
)

In [0]:
silver_customer_df = (
    bronze_customer_df
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.col("company_name")).alias("company_name"),
        F.trim(F.col("industry")).alias("industry"),
        F.col("annual_sales").cast(LongType()).alias("annual_sales"),
        F.col("employee_count").cast(IntegerType()).alias("employee_count"),
        F.trim(F.col("branch_name")).alias("branch_name"),
        F.trim(F.col("relationship_manager")).alias(
            "relationship_manager"
        ),
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

In [0]:
(
    silver_customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.silver_customer")
)

In [0]:
bronze_transaction_df = spark.table(
    "workspace.bank.bronze_transaction_summary"
)

In [0]:
silver_transaction_monthly_df = (
    bronze_transaction_df
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.col("month").alias("transaction_month"),
        F.col("monthly_inflow").cast("long").alias("monthly_inflow"),
        F.col("monthly_outflow").cast("long").alias("monthly_outflow"),
        F.col("deposit_balance").cast("long").alias("deposit_balance"),
        F.col("loan_balance").cast("long").alias("loan_balance"),
        F.col("overdue_days").cast("int").alias("overdue_days"),
    )
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("transaction_month").isNotNull())
    .dropDuplicates([
        "customer_id",
        "transaction_month"
    ])
)

In [0]:
(
    silver_transaction_monthly_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.bank.silver_transaction_monthly"
    )
)

In [0]:
bronze_crm_df = spark.table(
    "workspace.bank.bronze_crm_activity"
)

In [0]:
silver_crm_activity_df = (
    bronze_crm_df
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.to_date(F.col("activity_date")).alias("activity_date"),
        F.trim(F.col("meeting_note")).alias("meeting_note"),
        F.trim(F.col("customer_concern")).alias(
            "customer_concern"
        ),
        F.trim(F.col("next_action")).alias("next_action"),
    )
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("activity_date").isNotNull())
    .dropDuplicates(
        ["customer_id", "activity_date", "meeting_note"]
    )
)

In [0]:
(
    silver_crm_activity_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.silver_crm_activity")
)

In [0]:
%sql
SELECT
    'silver_customer' AS table_name,
    COUNT(*) AS row_count
FROM workspace.bank.silver_customer

UNION ALL

SELECT
    'silver_transaction_monthly',
    COUNT(*)
FROM workspace.bank.silver_transaction_monthly

UNION ALL

SELECT
    'silver_crm_activity',
    COUNT(*)
FROM workspace.bank.silver_crm_activity;

## ゴールドテーブル作成

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customer_df = spark.table(
    "workspace.bank.silver_customer"
)

transaction_df = spark.table(
    "workspace.bank.silver_transaction_monthly"
)

crm_df = spark.table(
    "workspace.bank.silver_crm_activity"
)

In [0]:
print("customer:", customer_df.count())
print("transaction:", transaction_df.count())
print("crm:", crm_df.count())

display(
    transaction_df.agg(
        F.min("transaction_month").alias("min_month"),
        F.max("transaction_month").alias("max_month")
    )
)

In [0]:
latest_month_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("transaction_month").desc())
)

oldest_month_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("transaction_month").asc())
)

transaction_ranked_df = (
    transaction_df
    .withColumn(
        "latest_month_rank",
        F.row_number().over(latest_month_window)
    )
    .withColumn(
        "oldest_month_rank",
        F.row_number().over(oldest_month_window)
    )
)

display(transaction_ranked_df)

In [0]:
latest_transaction_df = (
    transaction_ranked_df
    .filter(F.col("latest_month_rank") == 1)
    .select(
        "customer_id",
        F.col("transaction_month").alias("latest_transaction_month"),
        F.col("monthly_inflow").alias("latest_monthly_inflow"),
        F.col("monthly_outflow").alias("latest_monthly_outflow"),
        F.col("deposit_balance").alias("latest_deposit_balance"),
        F.col("loan_balance").alias("latest_loan_balance"),
        F.col("overdue_days").alias("latest_overdue_days")
    )
)

display(latest_transaction_df)

In [0]:
oldest_transaction_df = (
    transaction_ranked_df
    .filter(F.col("oldest_month_rank") == 1)
    .select(
        "customer_id",
        F.col("transaction_month").alias("oldest_transaction_month"),
        F.col("monthly_inflow").alias("oldest_monthly_inflow"),
        F.col("deposit_balance").alias("oldest_deposit_balance")
    )
)

display(oldest_transaction_df)

In [0]:
transaction_feature_df = (
    latest_transaction_df
    .join(
        oldest_transaction_df,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "inflow_change_rate",
        F.when(
            F.col("oldest_monthly_inflow") > 0,
            (
                F.col("latest_monthly_inflow")
                - F.col("oldest_monthly_inflow")
            ) / F.col("oldest_monthly_inflow")
        )
    )
    .withColumn(
        "deposit_change_rate",
        F.when(
            F.col("oldest_deposit_balance") > 0,
            (
                F.col("latest_deposit_balance")
                - F.col("oldest_deposit_balance")
            ) / F.col("oldest_deposit_balance")
        )
    )
    .withColumn(
        "net_cash_flow",
        F.col("latest_monthly_inflow")
        - F.col("latest_monthly_outflow")
    )
    .withColumn(
        "loan_to_deposit_ratio",
        F.when(
            F.col("latest_deposit_balance") > 0,
            F.col("latest_loan_balance")
            / F.col("latest_deposit_balance")
        )
    )
)

display(transaction_feature_df)

In [0]:
latest_crm_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("activity_date").desc())
)

latest_crm_df = (
    crm_df
    .withColumn(
        "crm_rank",
        F.row_number().over(latest_crm_window)
    )
    .filter(F.col("crm_rank") == 1)
    .select(
        "customer_id",
        F.col("activity_date").alias("last_activity_date"),
        F.col("meeting_note").alias("latest_meeting_note"),
        F.col("customer_concern").alias("latest_customer_concern"),
        F.col("next_action").alias("latest_next_action")
    )
)

display(latest_crm_df)

In [0]:
latest_crm_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("activity_date").desc())
)

latest_crm_df = (
    crm_df
    .withColumn(
        "crm_rank",
        F.row_number().over(latest_crm_window)
    )
    .filter(F.col("crm_rank") == 1)
    .select(
        "customer_id",
        F.col("activity_date").alias("last_activity_date"),
        F.col("meeting_note").alias("latest_meeting_note"),
        F.col("customer_concern").alias("latest_customer_concern"),
        F.col("next_action").alias("latest_next_action")
    )
)

display(latest_crm_df)


In [0]:
EVALUATION_DATE = "2026-07-20"

crm_feature_df = (
    latest_crm_df
    .withColumn(
        "days_since_last_activity",
        F.datediff(
            F.to_date(F.lit(EVALUATION_DATE)),
            F.col("last_activity_date")
        )
    )
)

In [0]:
gold_base_df = (
    customer_df
    .join(
        transaction_feature_df,
        on="customer_id",
        how="left"
    )
    .join(
        crm_feature_df,
        on="customer_id",
        how="left"
    )
)

display(gold_base_df)

In [0]:
gold_scored_df = (
    gold_base_df

    # 入金動向スコア
    .withColumn(
        "inflow_risk_score",
        F.when(F.col("inflow_change_rate") <= -0.30, 30)
        .when(F.col("inflow_change_rate") <= -0.15, 20)
        .when(F.col("inflow_change_rate") <= -0.05, 10)
        .otherwise(0)
    )

    # 預金残高スコア
    .withColumn(
        "deposit_risk_score",
        F.when(F.col("deposit_change_rate") <= -0.20, 25)
        .when(F.col("deposit_change_rate") <= -0.10, 15)
        .when(F.col("deposit_change_rate") <= -0.05, 8)
        .otherwise(0)
    )

    # 延滞スコア
    .withColumn(
        "overdue_risk_score",
        F.when(F.col("latest_overdue_days") >= 10, 25)
        .when(F.col("latest_overdue_days") >= 5, 15)
        .when(F.col("latest_overdue_days") >= 1, 8)
        .otherwise(0)
    )

    # 営業接点スコア
    .withColumn(
        "activity_risk_score",
        F.when(F.col("days_since_last_activity") >= 90, 20)
        .when(F.col("days_since_last_activity") >= 60, 12)
        .when(F.col("days_since_last_activity") >= 30, 5)
        .otherwise(0)
    )

    # 合計スコア
    .withColumn(
        "support_need_score",
        F.least(
            F.lit(100),
            F.col("inflow_risk_score")
            + F.col("deposit_risk_score")
            + F.col("overdue_risk_score")
            + F.col("activity_risk_score")
        )
    )
)

display(gold_scored_df)

In [0]:
gold_priority_df = (
    gold_scored_df
    .withColumn(
        "support_priority",
        F.when(F.col("support_need_score") >= 70, "High")
        .when(F.col("support_need_score") >= 40, "Medium")
        .otherwise("Low")
    )
)

display(
    gold_priority_df
    .select(
        "customer_id",
        "company_name",
        "support_need_score",
        "support_priority"
    )
    .orderBy(F.col("support_need_score").desc())
)

In [0]:
gold_reason_df = (
    gold_priority_df
    .withColumn(
        "support_reasons",
        F.concat_ws(
            " / ",

            F.when(
                F.col("inflow_change_rate") <= -0.15,
                F.concat(
                    F.lit("入金額が"),
                    F.format_number(
                        F.abs(F.col("inflow_change_rate") * 100),
                        1
                    ),
                    F.lit("%減少")
                )
            ),

            F.when(
                F.col("deposit_change_rate") <= -0.10,
                F.concat(
                    F.lit("預金残高が"),
                    F.format_number(
                        F.abs(F.col("deposit_change_rate") * 100),
                        1
                    ),
                    F.lit("%減少")
                )
            ),

            F.when(
                F.col("latest_overdue_days") > 0,
                F.concat(
                    F.lit("延滞"),
                    F.col("latest_overdue_days").cast("string"),
                    F.lit("日")
                )
            ),

            F.when(
                F.col("days_since_last_activity") >= 30,
                F.concat(
                    F.lit("最終営業接点から"),
                    F.col("days_since_last_activity").cast("string"),
                    F.lit("日経過")
                )
            )
        )
    )
)

display(
    gold_reason_df
    .select(
        "company_name",
        "support_need_score",
        "support_priority",
        "support_reasons"
    )
    .orderBy(F.col("support_need_score").desc())
)

In [0]:
gold_company_support_features_df = (
    gold_reason_df
    .select(
        # 顧客情報
        "customer_id",
        "company_name",
        "industry",
        "annual_sales",
        "employee_count",
        "branch_name",
        "relationship_manager",

        # 月次取引
        "latest_transaction_month",
        "latest_monthly_inflow",
        "latest_monthly_outflow",
        "net_cash_flow",
        "latest_deposit_balance",
        "latest_loan_balance",
        "loan_to_deposit_ratio",
        "latest_overdue_days",

        # 変化率
        "inflow_change_rate",
        "deposit_change_rate",

        # CRM
        "last_activity_date",
        "days_since_last_activity",
        "latest_meeting_note",
        "latest_customer_concern",
        "latest_next_action",

        # スコア
        "inflow_risk_score",
        "deposit_risk_score",
        "overdue_risk_score",
        "activity_risk_score",
        "support_need_score",
        "support_priority",
        "support_reasons"
    )
)

display(
    gold_company_support_features_df
    .orderBy(F.col("support_need_score").desc())
)

In [0]:
(
    gold_company_support_features_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.bank.gold_company_support_features"
    )
)

In [0]:
%sql
SELECT COUNT(*) AS company_count
FROM workspace.bank.gold_company_support_features;